In [ ]:
import SimpleITK as sitk
import numpy as np

# Load the colon mask
colon_mask = sitk.ReadImage(r'/path/to/workspace/classification_model_originalimage/totalsegment_work_resample/resampled_mask2.nii.gz')


In [ ]:
# Convert the colon mask to a numpy array
colon_array = sitk.GetArrayFromImage(colon_mask)  # Shape: [Z, Y, X]

# Find the indices where the colon mask is non-zero
colon_indices = np.nonzero(colon_array)

# Extract the Z coordinates
Y_coords = colon_indices[1]

# Calculate the minimum and maximum Z indices
min_Y = np.min(Y_coords)
# max_Y = np.max(Y_coords)
# print(min_Y)
# print(max_Y)

In [ ]:
# Get the spacing in the Z-direction (slice thickness)
voxel_height = colon_mask.GetSpacing()[1]  # In millimeters

# Define the physical height of the rectum area (e.g., 100 mm)
rectum_height_mm = 40  # Adjust based on anatomical knowledge

# Calculate the number of slices to include
num_voxels = int(rectum_height_mm / voxel_height)

# Define the upper Z limit for the rectum area
rectum_max_Y = min_Y + num_voxels
rectum_max_Y = min(rectum_max_Y, colon_array.shape[1] - 1)  # Ensure within bounds

print(rectum_max_Y)
print(num_voxels)
print(min_Y)

In [ ]:
# Initialize an array to hold the rectum mask
rectum_array = np.zeros_like(colon_array)

# Copy the relevant slices from the colon mask
rectum_array[:, min_Y:rectum_max_Y+1, :] = colon_array[:, min_Y:rectum_max_Y+1, :]


In [ ]:
# Convert the numpy array back to a SimpleITK image
rectum_mask = sitk.GetImageFromArray(rectum_array)

# Copy spatial information from the original colon mask
rectum_mask.CopyInformation(colon_mask)


In [ ]:
# Find the non-zero indices in the rectum mask
rectum_indices = np.nonzero(rectum_array)

# Get the minimum and maximum indices for each dimension
min_Z_rect, max_Z_rect = np.min(rectum_indices[0]), np.max(rectum_indices[0])
min_Y_rect, max_Y_rect = np.min(rectum_indices[1]), np.max(rectum_indices[1])
min_X_rect, max_X_rect = np.min(rectum_indices[2]), np.max(rectum_indices[2])


In [ ]:
# Define the expansion size (in voxels)
expansion = 10  # Adjust based on desired margin

# Get the dimensions of the image
image_size = colon_mask.GetSize()  # Returns (X, Y, Z)

# Expand the bounding box, ensuring it stays within image bounds
min_Z_rect = max(min_Z_rect - expansion, 0)
max_Z_rect = min(max_Z_rect + expansion, image_size[2] - 1)

min_Y_rect = max(min_Y_rect - expansion, 0)
max_Y_rect = min(max_Y_rect + expansion, image_size[1] - 1)

min_X_rect = max(min_X_rect - expansion, 0)
max_X_rect = min(max_X_rect + expansion, image_size[0] - 1)


In [ ]:
# Load the original MR image
mr_image = sitk.ReadImage('/path/to/workspace/classification_model_originalimage/totalsegment_work_resample/resampled_image.nii.gz')

# Define the size and index for the Region of Interest (ROI)
roi_size = [
    int(max_X_rect - min_X_rect + 1),
    int(max_Y_rect - min_Y_rect + 1),
    int(max_Z_rect - min_Z_rect + 1)
]

roi_index = [
    int(min_X_rect),
    int(min_Y_rect),
    int(min_Z_rect)
]

# Set up the RegionOfInterestImageFilter
roi_filter = sitk.RegionOfInterestImageFilter()
roi_filter.SetSize(roi_size)
roi_filter.SetIndex(roi_index)

# Extract the patch
rectum_patch = roi_filter.Execute(mr_image)

# Save the extracted patch
sitk.WriteImage(rectum_patch, '/path/to/workspace/classification_model_originalimage/totalsegment_work_resample/rectum_patch_female.nii.gz')
print('done')

